In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

Para crear los conjuntos de datos siguiendo un enfoque semisupervisado, se consideran las siguientes simulaciones:

* **Conjunto de entrenamiento (train)**: simulación espontánea
* **Conjunto de validación (validation)**: una simulación de ataque quitándole los intervalos formados por [pt_ataque, pt_ataque + 100]
* **Conjunto de prueba (test)**: otra simulación de ataque distinta a la de validación sin quitarle nada

## Funciónes auxiliares y variables

In [2]:
def add_interval_column(df, long_interval):
    # Definimos los límites de los intervalos
    bins = range(0, 30000 + long_interval, long_interval)  # 0 a 30000 ms en intervalos de long_interval ms
    # Agregamos una nueva columna al DataFrame con las etiquetas de los intervalos
    df.loc[:, 'interval'] = pd.cut(df['timestamps'], bins=bins, labels=False) + 1
    return df

In [3]:
TW = 1
file_path = "./data/spontaneous_30s_movie_one_resultados/spikes_0.csv"
file_path_FLO_validation = "./data/FLO_4instants_25neurons_30s_movie_one_resultados/spikes_0.csv"
file_path_FLO_test = "./data/FLO_4instants_25neurons_30s_movie_one_resultados/spikes_1.csv"

## Conversión a series temporales

In [4]:
# Espontáneo - train
df = pd.read_csv(file_path, delimiter=";")
df = add_interval_column(df, TW) 
time_series_train = df.groupby(['interval']).size().reset_index(name='spikes')
time_series_train

,interval,spikes
0,12,1
1,13,1
2,14,1
3,15,2
4,16,6
...,...,...
29984,29996,223
29985,29997,201
29986,29998,194
29987,29999,182


In [5]:
# Ataque 0 - validation
df_FLO_val = pd.read_csv(file_path_FLO_validation, delimiter=";")
df_FLO_val = add_interval_column(df_FLO_val, TW) 
time_series_val = df_FLO_val.groupby(['interval']).size().reset_index(name='spikes')
time_series_val

,interval,spikes
0,8,2
1,9,1
2,10,2
3,11,4
4,12,6
...,...,...
29988,29996,228
29989,29997,195
29990,29998,188
29991,29999,185


In [6]:
# Ataque 1 - test
df_FLO_test = pd.read_csv(file_path_FLO_test, delimiter=";")
df_FLO_test = add_interval_column(df_FLO_test, TW) 
time_series_test = df_FLO_test.groupby(['interval']).size().reset_index(name='spikes')
time_series_test

,interval,spikes
0,7,1
1,8,2
2,9,1
3,10,1
4,11,2
...,...,...
29989,29996,227
29990,29997,206
29991,29998,194
29992,29999,182


## Creación conjuntos de datos

In [7]:
#############################
########### TRAIN ###########
#############################

time_series_train.to_csv('./data/train_Movie.csv', index=False)

#############################
######## VALIDATION #########
#############################

# Definir rangos a eliminar
rangos_a_quitar = (
    ((time_series_val['interval'] >= 1) & (time_series_val['interval'] <= 100)) |
    ((time_series_val['interval'] >= 7500) & (time_series_val['interval'] <= 7600)) |
    ((time_series_val['interval'] >= 15000) & (time_series_val['interval'] <= 15100)) |
    ((time_series_val['interval'] >= 22500) & (time_series_val['interval'] <= 22600))
)

time_series_val_filtrado = time_series_val.loc[~rangos_a_quitar].reset_index(drop=True)
time_series_val_filtrado.to_csv("./data/validation_Movie.csv", index = False)

#############################
########### TEST ############
#############################

time_series_test.to_csv("./data/test_Movie.csv", index = False)